In [1]:
import pysam
import csv
from collections import defaultdict

In [9]:
# Inputs
bam_in_path = "../bam/D-hg38.pruned.bam"
barcode_to_donor_path = "../demuxlet/D-ATAC-barcode-indiv.tsv"
output_prefix = "D-donor_"  # e.g. donor_001.bam

In [10]:
# Read barcode-to-donor map
barcode_to_donor = {}
with open(barcode_to_donor_path) as f:
    reader = csv.reader(f, delimiter='\t')
    for barcode, donor in reader:
        # Barcodes may be stored with "-1" or not. Adjust as needed.
        barcode_to_donor[barcode.strip()] = donor.strip()

In [11]:
# Reverse map: donors → list of barcodes
donor_to_barcodes = defaultdict(set)
for bc, donor in barcode_to_donor.items():
    donor_to_barcodes[donor].add(bc)

In [12]:
# Create BAM writers per donor
bam_in = pysam.AlignmentFile(bam_in_path, "rb")
bam_out_files = {
    donor: pysam.AlignmentFile(f"{output_prefix}{donor}.bam", "wb", template=bam_in)
    for donor in donor_to_barcodes
}

In [13]:
# Iterate through BAM and write to the correct donor BAM file
for read in bam_in.fetch(until_eof=True):
    if read.has_tag("CB"):
        barcode = read.get_tag("CB")
        donor = barcode_to_donor.get(barcode)
        if donor and donor in bam_out_files:
            bam_out_files[donor].write(read)

In [14]:
# Close all files
bam_in.close()
for f in bam_out_files.values():
    f.close()

In [15]:
# Indexing
for donor in bam_out_files:
    pysam.index(f"{output_prefix}{donor}.bam")

In [ ]:
#move bams to bam_indiv folder since I didn't do thattttt